 Modeli konuştur. Sampling döngüsünü yaz, top-k ile örnek üret. Önce İngilizce bir başlangıç cümlesi dene, sonra Türkçe bir cümle. GPT-2'nin Türkçe çıktısı neden kötü? Aynı Türkçe cümle GPT-2'nin tokenizer'ında kaç token tutuyor? Geçen haftaki sonuçlarınla bağla. (0:31:00 - 0:45:50)


In [1]:
import torch
from torch.nn import functional as F
from transformers import GPT2LMHeadModel
import tiktoken

# Cihaz ayarı
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Kullanılan cihaz: {device}")

# GPT-2 (124M) modelini Hugging Face'ten yüklüyoruz
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
model.eval()

# GPT-2'nin BPE tokenizer'ı (tiktoken)
enc = tiktoken.get_encoding('gpt2')
print(f"Tokenizer hazır. Sözlük boyutu: {enc.n_vocab}")


Kullanılan cihaz: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Tokenizer hazır. Sözlük boyutu: 50257


### Top-K Sampling Döngüsü


In [6]:
def generate_text(prompt, max_length=40, top_k=50, num_return_sequences=3, seed=42):
    tokens = enc.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long, device=device)
    # Batch boyutu kadar çoğalt
    x = x.unsqueeze(0).repeat(num_return_sequences, 1)
    
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        
    while x.size(1) < max_length:
        with torch.no_grad():
            logits = model(x).logits # (B, T, vocab_size)
            logits = logits[:, -1, :] # son tokenın logitleri (B, vocab_size)
            probs = F.softmax(logits, dim=-1) # olasılıklara dönüştür
            
            # Top-k filtreleme. En yüksek 50 tokeni al
            topk_probs, topk_indices = torch.topk(probs, top_k, dim=-1) # (B, top_k)
            # Olasılıklara göre örnekle
            ix = torch.multinomial(topk_probs, 1) # (B, 1)
            # Seçilen indekse karşılık gelen gerçek token id'sini al
            xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
            # Diziye ekle
            x = torch.cat((x, xcol), dim=1)
            
    results = []
    for i in range(num_return_sequences):
        seq = x[i, :max_length].tolist()
        text = enc.decode(seq, errors='replace')
        results.append(text)
    return results


### İngilizce Başlangıç Cümlesi ile Deneme

In [3]:
prompt_en = "Hello, I'm a language model,"
samples_en = generate_text(prompt_en, max_length=45, top_k=50, num_return_sequences=3, seed=42)

print("=" * 70)
print(f"İNGİLİZCE PROMPT: '{prompt_en}'")
print("=" * 70)
for i, sample in enumerate(samples_en, 1):
    print(f"[Örnek {i}]:")
    print(sample)
    print("-" * 70)


İNGİLİZCE PROMPT: 'Hello, I'm a language model,'
[Örnek 1]:
Hello, I'm a language model, not a program.

So this morning I started studying for the interview in the lab. This was not a hard question, I decided to do it before I knew what I was
----------------------------------------------------------------------
[Örnek 2]:
Hello, I'm a language model, and one of the main things that bothers me when they create languages is how easy it becomes to create something that is difficult to understand. I find myself more confused about how to understand than
----------------------------------------------------------------------
[Örnek 3]:
Hello, I'm a language model, and I wrote it off on the grounds that a language model would make me more fluent. But I'm not sure if that's true.

But let's go back to your
----------------------------------------------------------------------


### Türkçe Başlangıç Cümlesi ile Deneme

In [4]:
prompt_tr = "Bugün hava çok güzel çünkü"
samples_tr = generate_text(prompt_tr, max_length=45, top_k=50, num_return_sequences=3, seed=42)

print("=" * 70)
print(f"TÜRKÇE PROMPT: '{prompt_tr}'")
print("=" * 70)
for i, sample in enumerate(samples_tr, 1):
    print(f"[Örnek {i}]:")
    print(sample)
    print("-" * 70)


TÜRKÇE PROMPT: 'Bugün hava çok güzel çünkü'
[Örnek 1]:
Bugün hava çok güzel çünkün kudükünn iurÖggüna havan åjör çünmün k
----------------------------------------------------------------------
[Örnek 2]:
Bugün hava çok güzel çünkünda łrın kontakte: (tok vos úleri veşşte güzel)
----------------------------------------------------------------------
[Örnek 3]:
Bugün hava çok güzel çünküntückt zem hin za vor kommen.

Büchel, G. (1946).
----------------------------------------------------------------------


### Tokenizer Analizi: Aynı Türkçe Cümle Kaç Token Tutuyor?


In [5]:
cumle_tr = "Bugün hava çok güzel çünkü dışarıda güneş parlıyor ve çocuklar parkta oyun oynuyor."
cumle_en = "Today the weather is very nice because the sun is shining outside and children are playing in the park."

tokens_tr = enc.encode(cumle_tr)
tokens_en = enc.encode(cumle_en)

bytes_tr = cumle_tr.encode('utf-8')
bytes_en = cumle_en.encode('utf-8')

print(f"{'Metrik':<30} | {'Türkçe Cümle':<25} | {'İngilizce Cümle':<25}")
print("-" * 86)
print(f"{'Karakter Sayısı':<30} | {len(cumle_tr):<25} | {len(cumle_en):<25}")
print(f"{'UTF-8 Byte Sayısı':<30} | {len(bytes_tr):<25} | {len(bytes_en):<25}")
print(f"{'GPT-2 Token Sayısı':<30} | {len(tokens_tr):<25} | {len(tokens_en):<25}")
print(f"{'Karakter / Token Oranı':<30} | {len(cumle_tr)/len(tokens_tr):<25.2f} | {len(cumle_en)/len(tokens_en):<25.2f}")
print(f"{'Byte / Token Oranı':<30} | {len(bytes_tr)/len(tokens_tr):<25.2f} | {len(bytes_en)/len(tokens_en):<25.2f}")

print("\n--- TÜRKÇE CÜMLENİN TOKEN'LARA PARÇALANIŞI ---")
tr_pieces = [enc.decode_single_token_bytes(t).decode('utf-8', errors='replace') for t in tokens_tr]
print(f"Token Sayısı: {len(tokens_tr)}")
print(f"Token Parçaları:\n{tr_pieces}")

print("\n--- İNGİLİZCE CÜMLENİN TOKEN'LARA PARÇALANIŞI ---")
en_pieces = [enc.decode_single_token_bytes(t).decode('utf-8', errors='replace') for t in tokens_en]
print(f"Token Sayısı: {len(tokens_en)}")
print(f"Token Parçaları:\n{en_pieces}")


Metrik                         | Türkçe Cümle              | İngilizce Cümle          
--------------------------------------------------------------------------------------
Karakter Sayısı                | 83                        | 103                      
UTF-8 Byte Sayısı              | 96                        | 103                      
GPT-2 Token Sayısı             | 47                        | 20                       
Karakter / Token Oranı         | 1.77                      | 5.15                     
Byte / Token Oranı             | 2.04                      | 5.15                     

--- TÜRKÇE CÜMLENİN TOKEN'LARA PARÇALANIŞI ---
Token Sayısı: 47
Token Parçaları:
['Bug', 'ü', 'n', ' ha', 'va', ' �', '�', 'ok', ' g', 'ü', 'zel', ' �', '�', 'ü', 'n', 'k', 'ü', ' d', 'ı', 'ş', 'ar', 'ı', 'da', ' g', 'ü', 'ne', 'ş', ' par', 'l', 'ı', 'y', 'or', ' ve', ' �', '�', 'oc', 'uk', 'lar', ' park', 'ta', ' oy', 'un', ' o', 'yn', 'uy', 'or', '.']

--- İNGİLİZCE CÜMLENİN TOKEN'LARA